# 5장 1강: 함수 정의와 매개변수

## [심화] 파이썬 함수의 메모리 생애주기 (Heap vs Call Stack)

#### Python 메모리
- `정적 메모리`: 한번 기록되면 제거되지 않음 -> 프로그램 실행 동안 비교적 오래 유지되는 영역
    - 코드 영역
        - 프로그램 실행에 필요한 코드 정보
        - 내장함수 ex) print(), int(), ...
        - 제어문
    - 전역변수 (Global 영역)   
        - 전역 이름 및 객체에 대한 참조 등

- `동적 메모리`: 실행 중 객체 생성과 소멸을 반복 (안쓰면 제거됨)
    - **Heap 영역 메모리**(객체가 주로 존재)
        - Object: ex) int, str, list, dict, 사용자 정의 객체 등
        - PyFunctionObject
            - PyCodeObject
            - 함수의 매개변수 정의 - 주소 존재 (calc라는 이름표를 가진 함수의 주소)
            - 수행 코드 정의

- `call stack` (스택 영역): 함수 호출 시 실행, 상태 관리 영역. '임시 메모리'
    ex) 함수 호출 -> frame 생성 -> 함수 실행에 필요한 정보 관리
    - PyFrameObject
    - 수행 완료시 frame을 메모리에서 제거

변수(객체를 가르키는 이름) -> 객체 -> reference (변수와 객체의 연결) -> stack frame -> heap
`x = 10`일 때
- stack frame: `x`라는 이름이 존재하는 실행 공간
- heap: `10` 객체가 존재하는 곳


전역변수: 함수 외부에 정의된 변수
지역변수: 함수 내부에 정의된 변수. 매개변수도 지역변수에 해당

*** Stack Memory ***
- whenver a function is called, Python creates a stack frame for the function and add it to the call stack.
- the local variables(references to objects) declared inside the function (local variable) are stored inside the function's stack frame
- **after** finishing executing, stack memory is freed automatically

*** Heap Memory ***
- a variable created -> its object/value is allocated in Heap memory ex) large data structures(lists, dictionaries, objects)
- only the reference of the object is stored in **stack** memory  
- objects in Heap memory can be shared among multiple functions even after a function finished executing
- Garbage Collection -> clean up unused object from Heap memeory (regarding reference counter) after the stack memory frees the function frame


In [151]:
def add(a, b):
    result = a + b
    return result

def calc():
    x = 10
    y = 20
    return add(x, y)


# mental model of memory process

# FUNCTION DEFINITION
#            ↓
#    Function object exists in heap/object memory
#            ↓
#    CALL calc()
#            ↓
#    ┌───────────────────────┐
#    │ calc() stack frame    │
#    │ x → object            │
#    │ y → object            │
#    └───────────────────────┘
#            ↓ calls
#    ┌───────────────────────┐
#    │ add() stack frame     │
#    │ a → object            │
#    │ b → object            │
#    │ result → object       │
#    └───────────────────────┘
#            ↓
#    add() finishes
#            ↓
#    add() frame removed
#            ↓
#    calc() continues
#            ↓
#    calc() finishes
#            ↓
#    calc() frame removed
###

#### stack vs. queue

*** `compile` 기반 - C, C++, Java, ... ***
- 소스코드 -> 컴파일 -> 기계어 -> 실행
- 함수를 코드 그 자체로 봄
- 정적 영역 메모리 중에서 일정 부분을 할당해서 그곳에 정의함

*** `interpreter` 기반 - Python, JavaScript ***
- 소스코드 -> 한줄씩 해석 while 실행중 -> 결과
- 함수 명세 객체 -> Heap 영역에 PyCodeObject, PyFunctionObject로 stored




객체.
interpreter 기반으로서  python memeory 동작과정.


In [152]:
x = 10
y = x
print(x is y) #같은 객체 10을 참조함
print(id(x)  == id(y))

True
True


In [153]:
def add(num1, num2): #매개변수==지역변수. 
    num3 = 30 #지역변수

    result = num1 + num2 + num3 # result = local variable

    return result 

result = add(10,20) # result = global variable

print(result)


#1. add라는 이름이 Heap memory에 있는 Function Object를 참조함.
#2. Function Object는 Code Object 및 함수 관련 정보를 가짐
#3. id(add)로 Function Object의 identity를 확인할 수 있음.
# num1, num2는 함수 호출시 Frame에서 지역 이름으로 관리됨.
# 10과 20은 Heah에 존재하는 int 객체



#      함수 내부                         함수 외부

#    add Frame                           Global
# ┌──────────────┐                  ┌──────────────┐
# │ num1 → 10    │                  │ result → 60  │
# │ num2 → 20    │                  └──────┬───────┘
# │ num3 → 30    │                         │
# │ result → 60  │                         ↓
# └──────────────┘                       Heap
#                                        60 객체

60


In [154]:
def func(num): # 실행 순서: func() -> add()
    result = add(20, 30)
    result += num
    return result

result = func(40)

print(result)

# call stack에서 함수별로 다른 PyFrameObject가 만들어짐
# 제거 시 마지막에 호출된 add()의 PyFrameObject가 먼저 제거됨 <- Stack (LIFO)

120


In [155]:
# int id
x = 10
y = x

x += 1

if id(x) != id(y):
    print("x and y do not refer to the same object")

# list id
a = [1, 2, 3]
b = a
print(f"a ID: {id(a)}, b ID: {id(b)}") # Same ID → both point to same list 
b.append(4) 
print(f"a: {a} b: {b}")

x and y do not refer to the same object
a ID: 2323802235712, b ID: 2323802235712
a: [1, 2, 3, 4] b: [1, 2, 3, 4]


## 1. 함수의 기본 정의와 작동 방식

#### 정의 방법
```
def func(parameter, ...):
    # 처리 부분
    return 반환값


func(argument)
```

#### 함수를 호출하는 방법 (실행하는 방법)
이름(...) -> 엔터

#### 목적
코드의 재활용

### 
define -> process -> return

### return
- 반환값 정의
- 함수의 종료

In [ ]:
def calc(x):
    y = x * 2 + 1
    return y

result = calc(10)
print(result)

In [ ]:
# 변수처럼 같은 함수에 이름표를 붙여 참조 가능
calc2 = calc
print(calc2(20))

In [ ]:
def add(num):

    if num == 999:
        print("종료")
        return
    
    result = num **2

    return result

add(999)

## 2. Parameter 매개변수 and Argument 인수

- 매개변수: 함수 호출시 입력 부분에서 값을 담아주는 변수
- 인수: 호출시 사용된 값 (10)

*** isinstance() ***
- isinstance(object, type)
- type can be a tuple of types or classes


*** enumerate() ***
- enumerate(iterbale, starting index number)
- brings value and index at the same time while iterating
- starting number default at 0

In [ ]:
# isinstance() example
x = isinstance("Hello", (str, int, float, dict, tuple, list))
print(x)

In [ ]:
# enumerate() example
fruits = ('apple', 'banana', 'cherry')
y = enumerate(fruits)
print(y)

for index, fruit in enumerate(fruits, start=2):
    print(index, fruit)

## 3. Positional Arguments 위치인수 and Keyword Arguments 키워드 인수

- `위치 인수`: 매개변수의 순서대로 값을 지정
- `키워드 인수`: 매개변수 명과 함께 인수의 이름을 지정, 매개변수명=값 형태로 전달. 이름이 중요, 순서는 상관 없음

`위치 인수`는 항상 `키워드 인수` 앞에 있어야 함

In [ ]:
def add(num1, num2):
    return num1 + num2 #변수 지정 하지 않고 바로 return하여 memory 절약 가능

In [ ]:
add(10, 20) # 위치 인수

In [ ]:
add(num1=10, num2=20) # 키워드 인수

In [ ]:
add(num2=20, num1=10) #키워드 인수는 순서 상관 x

In [ ]:
def add(num1, num2, name, mobile):
    return f"{num1 + num2}, {name=}, {mobile=}"

In [ ]:
add(10, 20, mobile="010-000-0000", name="김영희")

In [ ]:
add(mobile="010-000-0000", name="김영희", 10, 20) #positional argument cannot appear after keyword argument

## 4. Default Parameter 기본값 매개변수

- 매개변수에 값을 할당하지 않아도 기본적으로 가질 수 있는 값 설정
- 기본값이 있는 매개변수는 오른쪽 끝부터 차례대로 정의

In [ ]:
def add(num1, num2, name, mobile = "010"): # mobile = "010" 은 default parameter
    return f"{num1 + num2}, {name=}, {mobile=}"

In [87]:
add(10, 20, "김영희")

"30, name=None, mobile='김영희'"

In [ ]:
# def add(num1, num2, name = None, mobile):  --> default parameter는 오른쪽 끝으로 이동시키기
#    return f"{num1 + num2}, {name=}, {mobile=}"

def add(num1, num2, mobile, name = None):
    return f"{num1 + num2}, {name=}, {mobile=}"

In [86]:
add(10, 20, "010-0000-0000")

"30, name=None, mobile='010-0000-0000'"

## 5. 가변 인수(*arg)와 키워드 가변 인수(**kwargs)

- `*arg`: 가변적인 위치 인수, 매개변수 갯수 상관없이 지정 가능, 키워드 매개변수로는 사용 불가(이름을 알 수 없음), **kwargs보다 앞에 위치
- `**kwarg`: 

In [112]:
def add(*nums):
    #print(numbers, type(numbers))
    #return sum(numbers)
    total = 0
    for num in nums:
        total += num

    return total

print(add(1, 2, 3))
print(add(1, 2, 3, 4))
print(add(1, 2, 3, 4, 5))

6
10
15


In [ ]:
def add(*nums):
    total = 0
    for i in range(len(nums)):
        total += nums[i]
        print(f"{i+1}번째 {total}")

    return total


print(add(1, 2, 3))
print(add(1, 2, 3, 4))
print(add(1, 2, 3, 4, 5))

1번째 1
2번째 3
3번째 6
6
1번째 1
2번째 3
3번째 6
4번째 10
10
1번째 1
2번째 3
3번째 6
4번째 10
5번째 15
15


In [116]:
def print_user_info(**user):
    print(user)


print_user_info(username= "Emily", phone= "010-0000-0000")

{'username': 'Emily', 'phone': '010-0000-0000'}


In [117]:
def print_user_info(name, age, **info):
    print("name:", name)
    print("age:", age)
    print(info)


print_user_info("김철수", 20, mobie="010-0000-0000")

name: 김철수
age: 20
{'mobie': '010-0000-0000'}


In [ ]:
def print_user_info(name, age, *args, **kwargs):
    print("name:", name)
    print("age:",  age)
    print(args)
    print(kwargs)

print_user_info("김철수", 20, 'A', 20, mobie="010-0000-0000") # 'A', 20은 Tuple로 *args에 들어감

name: 김철수
age: 20
('A', 20)
{'mobie': '010-0000-0000'}


## 6. return 문의 역할과 복수 데이터 반환

In [124]:
def calc(num1, num2):
    return num1 + num2, num1 * num2, num1 - num2, num1 / num2 # return 값 여러개면 튜플 하나로 반환

print(calc(2,2))
x = a, b, c, d = calc(2,2)
print(x)


(4, 4, 0, 1.0)
(4, 4, 0, 1.0)


In [125]:
import sys

def outer_func():
    x = 10

    def inner_func():
        y = 20

        # 1. 현재 실행 중인 프레임 객체 가져오기
        frame = sys._getframe()
        print(f"1. 현재 실행 함수: {frame.f_code.co_name}")
        print(f"2. 현재 지역 변수(f_locals): {frame.f_locals}")
        print(f"3. 전역 스코프 참조 여부: {'outer_func' in frame.f_globals}")

        # 2. f_back을 통해 부모 프레임(outer_func)으로 이동
        parent_frame = frame.f_back
        print(f"4. 부모 함수 이름: {parent_frame.f_code.co_name}")
        print(f"5. 부모 함수의 지역 변수: {parent_frame.f_locals}")

    inner_func()

outer_func()

1. 현재 실행 함수: inner_func
2. 현재 지역 변수(f_locals): {'y': 20, 'frame': <frame at 0x0000021D0D657880, file 'C:\\Users\\myung\\AppData\\Local\\Temp\\ipykernel_25548\\607032025.py', line 12, code inner_func>}
3. 전역 스코프 참조 여부: True
4. 부모 함수 이름: outer_func
5. 부모 함수의 지역 변수: {'x': 10, 'inner_func': <function outer_func.<locals>.inner_func at 0x0000021D0D8ACEB0>}
